# EDA dataset latih

Data dari `data/processed/train.parquet`, hasil `python -m padamfirst.model.dataset`. Satu baris adalah satu sel H3 resolusi 7 yang punya titik api di hari itu (Juni sampai November 2015 sampai 2025). Label 1 kalau dalam 48 jam sel itu masih terbakar atau sel tetangga ring 1 mulai terbakar.

Jalankan ulang dari `pipeline/`:

```bash
uv run --group notebook jupyter nbconvert --to notebook --execute --inplace notebooks/eda.ipynb
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Paket padamfirst tidak di-install, notebook dijalankan dari pipeline/notebooks/.
sys.path.insert(0, str(Path.cwd().parent))
from padamfirst.model.dataset import FEATURES, OUT_PATH, summary

data = pd.read_parquet(OUT_PATH)
numeric = [f for f in FEATURES if f != "landcover_class"]
print(f"{len(data):,} baris, {data['h3_index'].nunique():,} sel, {len(FEATURES)} fitur")

## Jumlah baris dan proporsi label per tahun

Split PRD: latih 2015 sampai 2022, validasi 2023, uji 2024 sampai 2025.

In [ ]:
per_year = summary(data)
per_year["split"] = np.select(
    [per_year.index <= 2022, per_year.index == 2023], ["latih", "validasi"], "uji"
)
per_year.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(per_year.index.astype(str), per_year["label_rate"], color="#fd8d3c")
ax.axhline(data["label"].mean(), color="#555", linestyle="--", linewidth=1, label="rata-rata")
ax.set_ylabel("proporsi label 1")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## Nilai kosong per fitur

In [ ]:
empty = data[FEATURES].isna().mean()
empty[empty > 0].round(4).to_frame("porsi kosong")

## Sebaran fitur menurut label

Histogram ternormalisasi, sumbu x dipotong di persentil 1 dan 99 supaya ekor panjang (FRP, jumlah titik) tidak menutupi bentuknya.

In [ ]:
cols = 5
rows = int(np.ceil(len(numeric) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 2.4))
for ax, feature in zip(axes.flat, numeric, strict=False):
    values = data[feature].dropna()
    low, high = values.quantile([0.01, 0.99])
    bins = np.linspace(low, high, 30) if high > low else 10
    for label, color in [(0, "#3182bd"), (1, "#e6550d")]:
        part = data.loc[data["label"] == label, feature].dropna()
        clipped = part.clip(low, high)
        ax.hist(clipped, bins=bins, density=True, alpha=0.5, color=color, label=str(label))
    ax.set_title(feature, fontsize=9)
    ax.tick_params(labelsize=7)
    ax.set_yticks([])
for ax in axes.flat[len(numeric) :]:
    ax.axis("off")
axes.flat[0].legend(title="label", fontsize=7, frameon=False)
fig.tight_layout()
plt.show()

## Proporsi label per tutupan lahan

In [ ]:
landcover = data.groupby("landcover_class")["label"].agg(rows="size", label_rate="mean")
landcover.sort_values("rows", ascending=False).round(3)

## Korelasi

Spearman, karena banyak fitur miring dan hubungannya belum tentu linear. Kolom terakhir adalah korelasi tiap fitur dengan label.

In [ ]:
corr = data[[*numeric, "label"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(11, 9))
image = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=8)
fig.colorbar(image, shrink=0.7)
plt.show()

In [ ]:
with_label = corr["label"].drop("label").sort_values(key=abs, ascending=False)
with_label.round(3).to_frame("spearman dengan label")